In [21]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import json
from dataclasses import dataclass
from langgraph.graph import MessagesState, START, END, StateGraph
from langgraph.prebuilt import ToolNode

load_dotenv()

True

In [17]:
llm = ChatOpenAI(
    model=f"gpt://{os.getenv('YC_FOLDER_ID')}/yandexgpt/latest",
    base_url="https://llm.api.cloud.yandex.net/v1",
    api_key=os.getenv("YC_AI_API_KEY"),
    default_headers={"OpenAI-Project": os.getenv("YC_FOLDER_ID")},
    temperature=0.1,
)


In [20]:
@dataclass
class AgentState:
    messages : list

In [ ]:

def create_react_agent(tools_list=None, system_prompt=None):
    """Creates a ReAct agent with the given tools.

    Includes an explicit THOUGHT step: if the model calls a tool
    without writing reasoning text (common with function-calling),
    we make an extra LLM call to extract the reasoning.
    """
    # if tools_list is None:
    #     tools_list = ALL_TOOLS
    # if system_prompt is None:
    #     system_prompt = REACT_SYSTEM_PROMPT

    # parallel_tool_calls=False — forces the model to call one tool per step,
    # making the TAO loop explicit: Thought → Action → Observation → Thought → ...
    llm_with_tools = llm.bind_tools(tools_list, parallel_tool_calls=False)

    def agent_node(state: MessagesState):
        messages = state['messages']
        # Add system prompt if not present yet
        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages


        response = llm_with_tools.invoke(messages)

        if response.tool_calls and not response.content:
            tool_info = ', '.join(tc['name'] for tc in response.tool_calls)
            thought = llm.invoke(
                messages
                + [
                    HumanMessage(
                        content=f'You chose to call: {tool_info}. '
                        'In 1 sentence, explain why this is the right next step. '
                        'Reply with ONLY your reasoning, no tool calls.'
                    )
                ]
            )
            response.content = thought.content

        return {'messages': [response]}

    def should_continue(state: MessagesState):
        last_message = state['messages'][-1]
        if last_message.tool_calls:
            return 'tools'
        return END

    # Build the graph
    workflow = StateGraph(MessagesState)
    workflow.add_node('agent', agent_node)
    workflow.add_node('tools', ToolNode(tools_list))

    workflow.add_edge(START, 'agent')
    workflow.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})
    workflow.add_edge('tools', 'agent')

    return workflow.compile()


react_agent = create_react_agent()
print('ReAct agent created')

In [15]:
@tool
def add(a : int, b :int) -> str:
    """Складывает два целых числа a и b."""
    if not a:
        return json.dumps({
            'status' : 'error',
            'message' : f'Func call without attr: {a}'
        })
    if not b:
        return json.dumps({
            'status' : 'error',
            'message' : f'Func call without attr: {b}'
        })
    return json.dumps({
        'status' : 'success',
        'message' : f'{a} + {b} = {a + b}'
    })

In [11]:
try:
    llm_with_tools = llm.bind_tools([add])
except Exception as e:
    print(f'Ошибка: {e}')

Ошибка: 


In [ ]:
def build_agent_graph(system_promt : str, tools : list):
    llm_with_tools = llm.bind_tools([add])
    workflow =
